# Role interval — feasibility 128 FIT images, frozen A, T4 ×2
Chỉ kiểm tra bộ suy diễn ngữ nghĩa từ caption. **Không huấn luyện.** Internet ON, GPU T4 ×2, Save & Run All.
Giữ checkpoint A và prompt gốc. Không cần UPT full cache, prompt JSON mới, CHAIR hoặc token API.
Đầu ra: 128 cặp greedy/sample, graph/interval và 30 quyết định để kiểm tra trên HTML offline.
Ngưỡng: ít nhất500 FIT/replay khả dụng và tách biệt;16/128 cặp có aG khác0;8/128 cặp có xử lý interval/binding ảnh hưởng quyết định.
Qua ngưỡng số vẫn phải audit30 phép suy diễn (≥27 đúng, không lỗi thiếu nhãn→false). Không tự chuyển sang train.
Cap30 phút cho hash/prepare/generate/score; cài môi trường/parser và xuất hình nằm ngoài cap. Đây là giới hạn dừng, không phải ETA.

In [ ]:
print('ROLE INTERVAL FEASIBILITY: START',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
COCO_IMAGES=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
STALE_ROOT=Path('/kaggle/input/notebooks/ducanh2403/stale')
OUTPUT=Path('/kaggle/working/role_interval_feasibility_128')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON):assert p.is_file(),p
for p in (VISUAL_CACHE,COCO_IMAGES,STALE_ROOT):assert p.is_dir(),p
assert not OUTPUT.exists(),'Preserve prior output; choose fresh OUTPUT/session.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-role-interval-feasibility-'))/'repo'
LOG = Path('/tmp/upt_hidden_role-interval-feasibility.log')
PROJECT_COMMIT = '64f0bf475d86f99e59564d29003adeb1109e7d07'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'role_logic.py': 'c1d8b42bac004519e3d13ef100f060b722de3ade9af28ddd9e13ec559dc5acca', 'role_parser.py': '6a5ef61e86dabc4fc7bcef3258705f3e34b08abc507d8c40ef0cb68b1536734f', 'role_review.py': 'de9c332905058e27f4f6ede3ecd10ac2f4d46675547e0e52049c141e0ea08754', 'run_role.py': '2fa95be25b8c5d2bbe4eaf78e8676b8794afd54eda438b1c1c46ff68ac227de5', 'test_role.py': '995c5e5ab49669a5c3437da9a894631b1b5e5519c07b627fa6a322131bb5c26e', 'role_metadata.json': '3c2b7826e0c6eec0023a0ad9a1d2fbd3f6f16b73a5555f313db7b902805ca218', 'diagnostic_data.py': '15c0eab863ad6500436d25372e34e3395266774838db39b2bdb8e402ff150312', 'hidden_data.py': '9285d02c5aa7660a5656654e2efe034941bc24d6f7c300958c06fcca6f0d43c7', 'pilot_data.py': '93fa39b1de581d774417840be4d5e0b2b8d52bbf9ee53b479adad2c60a45d3d7'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','numpy','Pillow','h5py','transformers==4.57.1','spacy==3.8.7'])
import urllib.request
sys.path.insert(0,str(REPO))
from role_parser import MODEL_URL,MODEL_SHA
WHEEL=REPO/'en_core_web_sm-3.8.0-py3-none-any.whl'
print('Downloading pinned dependency parser (~12MB), CPU parsing on Kaggle only.',flush=True)
urllib.request.urlretrieve(MODEL_URL,WHEEL)
assert hashlib.sha256(WHEEL.read_bytes()).hexdigest()==MODEL_SHA,'Parser wheel SHA mismatch.'
run([sys.executable,'-m','pip','install','-q','--no-deps',WHEEL])
import torch
assert torch.cuda.device_count()==2,'Select T4x2.'
run([sys.executable,'-u','test_role.py'],cwd=REPO)
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),images=str(COCO_IMAGES),stale_root=str(STALE_ROOT),output=str(OUTPUT),
    project_commit=PROJECT_COMMIT,implementation_sha256=SOURCE_HASHES)
CONFIG=Path('/tmp/role_interval_config.json')
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
try:
    run([sys.executable,'-u','run_role.py','prepare','--config',CONFIG],cwd=REPO)
    CONFIG=OUTPUT/'protocol.json'
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0',
        'run_role.py','worker','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','run_role.py','finish','--config',CONFIG],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        run([sys.executable,'-u','run_role.py','archive','--config',CONFIG],cwd=REPO)
assert (OUTPUT/'receipt.json').is_file(),'Incomplete. Send review ZIP with logs; do not use partial output as a result.'
print('SEND:',str(OUTPUT)+'_review.zip',flush=True)
print('READ:',OUTPUT/'summary.json',flush=True)
print('OFFLINE HUMAN REVIEW:',OUTPUT/'review/index.html',flush=True)
